# M5: 20 or 50 products, two training years, one Random Forest

Run in **Google Colab → CPU → Runtime → Run all**. Self-contained: no repository clone, credentials, Drive mount, or Store database. This downloads the public 116 MB M5 Parquet file, filters CA_1 / FOODS, samples 20 products using training data only (seed 42), and runs the exact bundled RetailMind forecasting functions.

Two full calendar training years precede 84 held-out days: two 28-day validation blocks and one final 28-day test. Expanding training windows may use observations from earlier blocks, never from the forecast block. The Random Forest recipe is fixed before evaluation: 15 trees, depth 8, seed 42; direct daily predictions to day 28 with no recursive actuals. Validation selects only the any-sale-day threshold. Do not tune against the final test.

**Research only.** Source sales do not establish lost demand, stock availability, or local Store accuracy. Original dates and anonymous product IDs stay unchanged. No inventory is modified. [Dataset](https://huggingface.co/datasets/skforecast/m5_daily), [shared design](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html).

Set PRODUCT_COUNT to 20 or 50 in the run cell. The 50-product run retains the original 20 and adds 30 training-eligible products with seed 42. Compare both runs on the original 20; all-50 averages describe a different population. The previously inspected final window is development evidence, not independent confirmation.


In [ ]:
# Pin experiment packages. If Colab requests a restart, restart then Run all.
%pip install -q numpy==2.4.4 pandas==2.3.3 scikit-learn==1.9.1 pyarrow==26.0.0 matplotlib==3.11.2


## Frozen shared source
These four files are embedded verbatim from this experiment branch so the notebook works without access to the private repository. Update the bundle if shared source changes; the repository test checks this. Only forecasting code is included, no operational data or configuration.


In [ ]:
BUNDLED_SOURCES = {'ml_cv.py': 'import numpy as np\nimport pandas as pd\nfrom typing import List, Tuple\n\ndef rolling_origin_split(df: pd.DataFrame, date_col: str, horizon_days: int, n_splits: int = 3) -> List[Tuple[pd.DataFrame, pd.DataFrame]]:\n    """\n    Creates rolling-origin chronological splits.\n    df must have date_col.\n    Returns list of (train_df, test_df)\n    """\n    df = df.sort_values(by=date_col)\n    dates = df[date_col].drop_duplicates().sort_values().tolist()\n    \n    splits = []\n    \n    # Evaluate the last n_splits periods of length horizon_days\n    # For example, if horizon_days=7 and n_splits=3, we need 21 days at the end for testing\n    \n    for i in range(n_splits):\n        # The test period is [end - (i+1)*horizon_days, end - i*horizon_days)\n        test_end_idx = len(dates) - i * horizon_days\n        test_start_idx = test_end_idx - horizon_days\n        \n        if test_start_idx < 1:\n            break # Not enough data\n            \n        test_start_date = dates[test_start_idx]\n        if test_end_idx < len(dates):\n            test_end_date = dates[test_end_idx]\n            test_mask = (df[date_col] >= test_start_date) & (df[date_col] < test_end_date)\n        else:\n            test_mask = (df[date_col] >= test_start_date)\n            \n        train_mask = df[date_col] < test_start_date\n        \n        splits.append((df[train_mask], df[test_mask]))\n        \n    return splits[::-1]  # Return in chronological order\n', 'ml_evaluation.py': 'import numpy as np\n\ndef calculate_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict:\n    mae = float(np.mean(np.abs(actual - predicted)))\n    rmse = float(np.sqrt(np.mean((actual - predicted)**2)))\n    \n    total_actual = float(np.sum(actual))\n    \n    if total_actual == 0:\n        wape = None\n        missing_metric_reason = "Zero actual demand"\n    else:\n        wape = float(np.sum(np.abs(actual - predicted)) / total_actual)\n        missing_metric_reason = None\n        \n    return {\n        \'mae\': mae,\n        \'rmse\': rmse,\n        \'wape\': wape,\n        \'missing_metric_reason\': missing_metric_reason\n    }\n\ndef seasonal_naive_forecast(history: np.ndarray, horizon: int, season_length: int = 7) -> np.ndarray:\n    """\n    Generate a seasonal naive forecast.\n    If there is enough history, use the same day from the previous season.\n    If not, use the last available value.\n    """\n    forecast = np.zeros(horizon)\n    for i in range(horizon):\n        # We need the value from `season_length` days ago.\n        # But we might be forecasting more than one season into the future.\n        # e.g., if forecasting day 8, we need day 1 of the forecast, which is day -6 of history.\n        \n        # Determine the historical index to pull from\n        # steps back from the *start* of the forecast window\n        steps_back = season_length - (i % season_length)\n        \n        if len(history) >= steps_back:\n            forecast[i] = history[-steps_back]\n        elif len(history) > 0:\n            forecast[i] = history[-1]\n        else:\n            forecast[i] = 0.0\n            \n    return forecast\n', 'feature_pipeline.py': '"""One cutoff-aware feature contract for observed daily demand."""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import OneHotEncoder\n\nfrom ml_cv import rolling_origin_split\nfrom ml_evaluation import calculate_metrics, seasonal_naive_forecast\n\nFEATURE_VERSION = "2.0.0"\nFEATURE_COLUMNS = [\n    "product_id", "horizon_day", "lag_1", "lag_7", "lag_14", "lag_28",\n    "rolling_mean_7", "rolling_std_7", "rolling_mean_28", "rolling_std_28",\n    "day_of_week", "day_of_month", "month", "is_weekend",\n]\nMODEL_PARAMETERS = {"n_estimators": 15, "max_depth": 8, "random_state": 42}\n\n\ndef normalize_history(df):\n    if "Store ID" in df and df["Store ID"].nunique() != 1:\n        raise ValueError("Select exactly one isolated dataset Store")\n    columns = ["date", "product_id", "units_sold"]\n    versioned = "available_date" in df\n    if versioned:\n        columns.append("available_date")\n    data = df.rename(columns={\n        "Date": "date", "Product ID": "product_id", "Units Sold": "units_sold",\n    })[columns].copy()\n    data["date"] = pd.to_datetime(data["date"], errors="raise")\n    data["units_sold"] = pd.to_numeric(data["units_sold"], errors="raise")\n    if data.empty or data[["date", "product_id"]].isna().any().any():\n        raise ValueError("No usable demand history or missing observations")\n    if (data.date != data.date.dt.normalize()).any():\n        raise ValueError("Demand history must use daily dates")\n    values = data.units_sold.dropna() if versioned else data.units_sold\n    if not np.isfinite(values).all() or (values < 0).any():\n        raise ValueError("Demand observations must be finite and nonnegative")\n    keys = ["product_id", "date"]\n    if versioned:\n        data["available_date"] = pd.to_datetime(data.available_date, errors="raise")\n        if (data.available_date.isna().any() or (data.available_date < data.date).any()\n                or (data.available_date != data.available_date.dt.normalize()).any()):\n            raise ValueError("Invalid observation availability date")\n        keys.append("available_date")\n    if data.duplicated(keys).any():\n        raise ValueError("Duplicate daily demand observations")\n    return data.sort_values(keys).reset_index(drop=True)\n\n\ndef history_at(df, cutoff_date):\n    """Latest known daily value; null revisions withdraw unusable observations."""\n    data = normalize_history(df)\n    cutoff = pd.Timestamp(cutoff_date)\n    data = data[data.date <= cutoff]\n    if "available_date" in data:\n        data = data[data.available_date <= cutoff].drop_duplicates(["product_id", "date"], keep="last")\n        data = data.dropna(subset=["units_sold"]).drop(columns="available_date")\n    if data.empty:\n        raise ValueError("No usable demand history at cutoff")\n    return data\n\n\ndef build_feature_matrix(df, cutoff_date, horizon_days=28):\n    """Lag 1 is the observed origin day; never fill missing days with demand."""\n    data = history_at(df, cutoff_date)\n    cutoff = pd.Timestamp(cutoff_date)\n    expected = pd.date_range(cutoff - pd.Timedelta(days=27), cutoff)\n    rows = []\n    for pid, group in data.groupby("product_id", sort=True):\n        window = group[group.date.isin(expected)]\n        if len(window) != 28:\n            raise ValueError(f"Product {pid} requires 28 consecutive observed days at {cutoff.date()}")\n        sales = window.units_sold.to_numpy(dtype=float)\n        lags = {f"lag_{lag}": float(sales[-lag]) for lag in (1, 7, 14, 28)}\n        rolling = {}\n        for days in (7, 28):\n            rolling[f"rolling_mean_{days}"] = float(np.mean(sales[-days:]))\n            rolling[f"rolling_std_{days}"] = float(np.std(sales[-days:]))\n        for h in range(1, horizon_days + 1):\n            target = cutoff + pd.Timedelta(days=h)\n            rows.append({\n                "product_id": pid, "origin_date": cutoff, "target_date": target,\n                "horizon_day": h, **lags, **rolling,\n                "day_of_week": target.dayofweek, "day_of_month": target.day,\n                "month": target.month, "is_weekend": int(target.dayofweek >= 5),\n            })\n    return pd.DataFrame(rows)\n\n\ndef extract_training_targets(df, cutoff_date, horizon_days=28, label_cutoff=None):\n    data = history_at(df, label_cutoff or pd.Timestamp(cutoff_date) + pd.Timedelta(days=horizon_days))\n    features = build_feature_matrix(df, cutoff_date, horizon_days)\n    targets = features.merge(data, left_on=["product_id", "target_date"],\n                             right_on=["product_id", "date"], how="left", validate="one_to_one")\n    if targets.units_sold.isna().any():\n        raise ValueError("Missing observed training targets")\n    return targets.units_sold\n\n\ndef build_training_rows(df, cutoff_date):\n    """Weekly origins, with all 28 labels observed by the training cutoff."""\n    data = history_at(df, cutoff_date)\n    cutoff = pd.Timestamp(cutoff_date)\n    data = data[data.date <= cutoff]\n    if data.empty:\n        raise ValueError("No usable demand history")\n    first_origin = data.groupby("product_id").date.min().max() + pd.Timedelta(days=27)\n    last_origin = min(cutoff, data.date.max()) - pd.Timedelta(days=28)\n    rows = []\n    for origin in pd.date_range(first_origin, last_origin, freq="7D"):\n        features = build_feature_matrix(df, origin)\n        features["units_sold"] = extract_training_targets(df, origin, label_cutoff=cutoff).to_numpy()\n        rows.append(features)\n    if not rows:\n        raise ValueError("At least 56 consecutive observed days are required for training")\n    return pd.concat(rows, ignore_index=True)\n\n\ndef _fit_model(rows):\n    model = make_pipeline(\n        ColumnTransformer([(\'product\', OneHotEncoder(), [\'product_id\'])], remainder=\'passthrough\'),\n        RandomForestRegressor(**MODEL_PARAMETERS),\n    )\n    return model.fit(rows[FEATURE_COLUMNS], rows.units_sold)\n\n\ndef train_and_evaluate(df):\n    """Three rolling 28-day holdouts; score per-product 7/14/28-day totals."""\n    history = normalize_history(df)\n    data = history_at(history, history.date.max())\n    training = build_training_rows(history, data.date.max())\n    folds = []\n    totals = {h: {"actual": [], "predicted": [], "baseline": []} for h in (7, 14, 28)}\n    for past, held_out in rolling_origin_split(data, "date", 28, n_splits=3):\n        origin = held_out.date.min() - pd.Timedelta(days=1)\n        # Purge entire origins whose final label was not observable at this cutoff.\n        fit_rows = (build_training_rows(history, origin) if "available_date" in history else\n                    training[training.origin_date + pd.Timedelta(days=28) <= origin])\n        if fit_rows.empty:\n            raise ValueError("Insufficient history for three chronological holdouts")\n        model = _fit_model(fit_rows)\n        features = build_feature_matrix(history, origin)\n        actual = extract_training_targets(history, origin).to_numpy().reshape(-1, 28)\n        predicted = model.predict(features[FEATURE_COLUMNS]).reshape(-1, 28)\n        baseline = np.array([\n            seasonal_naive_forecast(group.units_sold.to_numpy(), 28)\n            for _, group in history_at(history, origin).groupby("product_id", sort=True)\n        ])\n        fold = {\n            "origin": str(origin.date()),\n            "training_label_end": str(fit_rows.target_date.max().date()),\n            "training_rows": len(fit_rows),\n            "validation_start": str((origin + pd.Timedelta(days=1)).date()),\n            "validation_end": str((origin + pd.Timedelta(days=28)).date()),\n            "evaluations": {},\n        }\n        for horizon in totals:\n            a = actual[:, :horizon].sum(axis=1)\n            p = predicted[:, :horizon].sum(axis=1)\n            b = baseline[:, :horizon].sum(axis=1)\n            metrics = calculate_metrics(a, p)\n            metrics.update({f"baseline_{k}": v for k, v in calculate_metrics(a, b).items()})\n            fold["evaluations"][str(horizon)] = metrics\n            for key, values in (("actual", a), ("predicted", p), ("baseline", b)):\n                totals[horizon][key].extend(values)\n        folds.append(fold)\n    if len(folds) != 3:\n        raise ValueError("Insufficient history for three chronological holdouts")\n    evaluations = {}\n    for horizon, values in totals.items():\n        actual = np.array(values["actual"])\n        metrics = calculate_metrics(actual, np.array(values["predicted"]))\n        metrics.update({f"baseline_{k}": v for k, v in\n                        calculate_metrics(actual, np.array(values["baseline"])).items()})\n        evaluations[horizon] = metrics\n    model = _fit_model(training)\n    return {\n        "model": model,\n        "evaluations": evaluations,\n        "metadata": {\n            "feature_version": FEATURE_VERSION, "features": FEATURE_COLUMNS,\n            "parameters": MODEL_PARAMETERS, "training_rows": len(training),\n            "training_window_start": str((training.origin_date.min() - pd.Timedelta(days=27)).date()),\n            "training_window_end": str(training.target_date.max().date()),\n            "data_window_start": str(data.date.min().date()),\n            "data_window_end": str(data.date.max().date()),\n            "products": data.product_id.unique().tolist(),\n            "observations": len(data), "folds": folds,\n        },\n    }\n', 'm5_sales_experiment.py': '"""Offline M5 experiment; never imports public sales into the Store database."""\nimport argparse\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\n\nfrom feature_pipeline import (\n    FEATURE_COLUMNS, FEATURE_VERSION, MODEL_PARAMETERS, _fit_model,\n    build_feature_matrix, build_training_rows, extract_training_targets, normalize_history,\n)\nfrom ml_cv import rolling_origin_split\nfrom ml_evaluation import calculate_metrics, seasonal_naive_forecast\n\nSOURCE_REVISION = "9eee585c46882aa28aaaf43733a5c55c3e32f058"\nSOURCE_URL = f"https://huggingface.co/datasets/skforecast/m5_daily/resolve/{SOURCE_REVISION}/m5_daily.parquet"\n\n\ndef prepare_history(raw, product_count=20):\n    """Two calendar years before three 28-day holdouts; sample using past only."""\n    data = raw.loc[(raw.store_id == "CA_1") & (raw.cat_id == "FOODS"),\n                   ["datetime", "item_id", "sales"]].rename(columns={\n                       "datetime": "date", "item_id": "product_id", "sales": "units_sold"})\n    data["product_id"] = data.product_id.astype("string")\n    data = normalize_history(data)\n    if (data.units_sold % 1 != 0).any():\n        raise ValueError("M5 sales must be whole units")\n    end = data.date.max()\n    training_end = end - pd.Timedelta(days=84)\n    start = training_end - pd.DateOffset(years=2) + pd.Timedelta(days=1)\n    past = data[data.date.between(start, training_end)]\n    days = (training_end - start).days + 1\n    eligible = past.groupby("product_id").agg(days=("date", "size"), units=("units_sold", "sum"))\n    codes = eligible.index[(eligible.days == days) & (eligible.units > 0)].sort_values()\n    if product_count not in (20, 50):\n        raise ValueError("Choose 20 or 50 products for the matched expansion experiment")\n    if len(codes) < product_count:\n        raise ValueError(f"Need {product_count} products with two complete training years and positive training sales")\n    selected = sorted(np.random.default_rng(42).choice(codes.to_numpy(), 20, replace=False).tolist())\n    if product_count == 50:\n        remaining = codes[~codes.isin(selected)].to_numpy()\n        selected = sorted(selected + np.random.default_rng(42).choice(remaining, 30, replace=False).tolist())\n    history = data[data.product_id.isin(selected) & data.date.between(start, end)].copy()\n    expected_days = (end - start).days + 1\n    if not history.groupby("product_id").size().eq(expected_days).all():\n        raise ValueError("Selected products have missing held-out days; do not silently resample")\n    return history, {\n        "source_url": SOURCE_URL, "source_revision": SOURCE_REVISION,\n        "store_id": "CA_1", "category": "FOODS", "seed": 42,\n        "products": selected, "eligible_products": len(codes),\n        "selection_rule": "Seeded sample of 20 training-eligible products; expansion retains those 20 and adds 30 seeded remaining products" if product_count == 50 else "Seeded uniform sample of sorted products complete and positive in training only",\n        "training_start": str(start.date()), "initial_training_end": str(training_end.date()),\n        "data_end": str(end.date()), "initial_training_days": days,\n        "observations": len(history),\n        "target": "Recorded sales units, not unconstrained demand",\n    }\n\n\ndef sale_day_metrics(actual, predicted, threshold):\n    actual = np.asarray(actual) > 0\n    predicted = np.asarray(predicted) >= threshold\n    tp = int((actual & predicted).sum())\n    fp = int((~actual & predicted).sum())\n    fn = int((actual & ~predicted).sum())\n    tn = int((~actual & ~predicted).sum())\n    return {\n        "tp": tp, "fp": fp, "fn": fn, "tn": tn,\n        "precision": tp / (tp + fp) if tp + fp else None,\n        "recall": tp / (tp + fn) if tp + fn else None,\n        "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,\n    }\n\n\ndef evaluate(history):\n    """Frozen recipe: first two blocks validate tau; last block is the final test."""\n    history = normalize_history(history)\n    training = build_training_rows(history, history.date.max())\n    predictions, folds = [], []\n    for number, (_, held_out) in enumerate(rolling_origin_split(history, "date", 28, 3), 1):\n        origin = held_out.date.min() - pd.Timedelta(days=1)\n        rows = training[training.origin_date + pd.Timedelta(days=28) <= origin]\n        if rows.empty:\n            raise ValueError("Insufficient training before holdout")\n        model = _fit_model(rows)\n        features = build_feature_matrix(history, origin)\n        frame = features[["product_id", "target_date", "horizon_day"]].copy()\n        frame["actual"] = extract_training_targets(history, origin).to_numpy()\n        frame["predicted"] = model.predict(features[FEATURE_COLUMNS])\n        frame["baseline"] = np.concatenate([\n            seasonal_naive_forecast(group.units_sold.to_numpy(), 28)\n            for _, group in history[history.date <= origin].groupby("product_id", sort=True)\n        ])\n        if not np.isfinite(frame[["actual", "predicted", "baseline"]]).all().all():\n            raise ValueError("Nonfinite experiment values")\n        frame["fold"] = number\n        frame["split"] = "test" if number == 3 else "validation"\n        predictions.append(frame)\n        folds.append({"fold": number, "origin": str(origin.date()),\n                      "training_label_end": str(rows.target_date.max().date()),\n                      "training_rows": len(rows), "split": frame.split.iloc[0]})\n        print(f"Finished {frame.split.iloc[0]} fold {number}, origin {origin.date()}", flush=True)\n    daily = pd.concat(predictions, ignore_index=True)\n    validation = daily[daily.split == "validation"]\n    thresholds = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]\n    threshold = max(thresholds, key=lambda tau: (\n        sale_day_metrics(validation.actual, validation.predicted, tau)["f1"] or 0))\n    scores = []\n    for (fold, split), group in daily.groupby(["fold", "split"]):\n        for horizon in (7, 14, 28):\n            totals = group[group.horizon_day <= horizon].groupby("product_id")[[\n                "actual", "predicted", "baseline"]].sum()\n            for pid, values in [("ALL", totals), *[(pid, row.to_frame().T) for pid, row in totals.iterrows()]]:\n                for name in ("predicted", "baseline"):\n                    scores.append({"fold": int(fold), "split": split, "horizon": horizon,\n                                   "product_id": pid, "model": "random_forest" if name == "predicted" else "seasonal_naive",\n                                   **calculate_metrics(values.actual.to_numpy(), values[name].to_numpy())})\n    classification = {}\n    for split, group in daily.groupby("split"):\n        classification[split] = {\n            "random_forest": sale_day_metrics(group.actual, group.predicted, threshold),\n            "seasonal_naive": sale_day_metrics(group.actual, group.baseline, 0.5),\n            "always_sale": sale_day_metrics(group.actual, np.ones(len(group)), 0.5),\n        }\n    return daily, pd.DataFrame(scores), {"folds": folds, "sale_day_threshold": threshold,\n                                         "classification": classification}\n\n\ndef write_discussion(scores, report, output):\n    lines = ["# M5 forecasting experiment results", "",\n             f"Runtime: {report[\'runtime\']}. Dataset revision: `{SOURCE_REVISION}`.",\n             f"Initial training: {report[\'training_start\']}–{report[\'initial_training_end\']} "\n             f"({report[\'initial_training_days\']} days), {len(report[\'products\'])} CA_1 FOODS products.",\n             "First two 28-day blocks validate the sale-day threshold; the final 28 days are the test.", "",\n             "| Test horizon | RF MAE (units) | RF RMSE (units) | RF WAPE | Baseline WAPE |",\n             "|---|---:|---:|---:|---:|"]\n    test = scores[(scores.split == "test") & (scores.product_id == "ALL")]\n    for h in (7, 14, 28):\n        group = test[test.horizon == h].set_index("model")\n        rf, base = group.loc["random_forest"], group.loc["seasonal_naive"]\n        percent = lambda value: "undefined" if pd.isna(value) else f"{100 * value:.2f}%"\n        lines.append(f"| {h} days | {rf.mae:.3f} | {rf.rmse:.3f} | {percent(rf.wape)} | {percent(base.wape)} |")\n    lines += ["", "MAE/RMSE above measure per-product horizon totals, not daily errors. "\n              "WAPE is an error fraction displayed as a percentage, not an accuracy score.", "",\n              f"Any-sale-day classification (threshold {report[\'sale_day_threshold\']}, chosen on validation only):"]\n    for name, metrics in report["classification"]["test"].items():\n        lines.append(f"- {name}: `{json.dumps(metrics)}`")\n    per_product = scores[(scores.split == "test") & (scores.product_id != "ALL") & (scores.horizon == 28)]\n    comparison = per_product.pivot(index="product_id", columns="model", values="mae")\n    losses = int((comparison.random_forest >= comparison.seasonal_naive).sum())\n    lines += ["", "## Interpretation and weaknesses", "",\n              f"Random Forest fails to improve 28-day absolute error over baseline for {losses}/{len(report[\'products\'])} products. "\n              "Inspect per-product scores; aggregate improvement can hide weak products.",\n              "High sale-day F1 can come from frequently positive sales. Compare with always_sale; "\n              "it does not establish quantity accuracy or stockout detection.",\n              "Only one final 28-day test window is available; two training years contain few annual cycles. "\n              "Walmart sales and US calendar effects do not establish local Store performance.",\n              "Recorded zeros do not identify stockouts or lost demand. The shared model currently uses "\n              "sales lags and calendar fields, not prices, promotions, or stock availability.", "",\n              "## Improvements to investigate", "",\n              "1. Inspect worst product/horizon errors and sustained over/underprediction in daily_predictions.csv.",\n              "2. Use validation to test price/promotion features only when known at the forecast origin; "\n              "do not use realized future prices or tune against this final test.",\n              "3. Evaluate additional future windows and locally collected Store history before operational approval.", "",\n              "## Integration boundary", "",\n              "Collect actual Store sales first. Train an isolated candidate when complete usable history "\n              "and chronological validation are sufficient; retain review/approval before daily forecast generation. "\n              "Public M5 history and models stay separate. Forecasts support replenishment and never change stock.",\n              "The existing readiness bands are below 56 days: Insufficient; 56–179: Low; 180–364: Medium; "\n              "365+: High only with acceptable validation. Minimum feature history is not sufficient "\n              "evidence for reliable validation or automatic approval.",\n              "No production activation or accuracy pass is inferred from this experiment."]\n    (output / "discussion.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("parquet", type=Path)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--products", type=int, choices=[20, 50], default=20)\n    parser.add_argument("--runtime", choices=["local Python", "Google Colab"], default="local Python")\n    args = parser.parse_args()\n    if args.output_dir.exists():\n        parser.error("Output directory already exists; choose a new experiment directory")\n    raw = pd.read_parquet(args.parquet, columns=["datetime", "item_id", "sales", "store_id", "cat_id"],\n                          filters=[("store_id", "==", "CA_1"), ("cat_id", "==", "FOODS")])\n    history, metadata = prepare_history(raw, args.products)\n    daily, scores, evaluation = evaluate(history)\n    report = {**metadata, **evaluation, "feature_version": FEATURE_VERSION,\n              "parameters": MODEL_PARAMETERS, "features": FEATURE_COLUMNS,\n              "runtime": args.runtime,\n              "software": {"python": platform.python_version(), "numpy": np.__version__,\n                           "pandas": pd.__version__, "scikit_learn": sklearn.__version__}}\n    with args.parquet.open("rb") as stream:\n        report["source_sha256"] = hashlib.file_digest(stream, "sha256").hexdigest()\n    args.output_dir.mkdir(parents=True, exist_ok=False)\n    history.to_csv(args.output_dir / "daily_history.csv", index=False)\n    daily.to_csv(args.output_dir / "daily_predictions.csv", index=False)\n    scores.to_csv(args.output_dir / "scores.csv", index=False)\n    (args.output_dir / "metrics.json").write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")\n    write_discussion(scores, report, args.output_dir)\n    print((args.output_dir / "discussion.md").read_text(encoding="utf-8"))\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, source in BUNDLED_SOURCES.items():
    from pathlib import Path
    work = Path('/content/retailmind_m5')
    work.mkdir(parents=True, exist_ok=True)
    (work / name).write_text(source, encoding='utf-8')
print('Shared forecasting source ready:', work)


In [ ]:
import sys
import urllib.request
sys.path.insert(0, str(work))
from m5_sales_experiment import SOURCE_URL
parquet = work / 'm5_daily.parquet'
if not parquet.exists():
    temporary = work / 'm5_daily.parquet.download'
    with urllib.request.urlopen(SOURCE_URL, timeout=120) as source, temporary.open('wb') as target:
        import shutil
        shutil.copyfileobj(source, target)
    temporary.replace(parquet)
print('Pinned source:', SOURCE_URL)
print('Downloaded bytes:', parquet.stat().st_size)


In [ ]:
import subprocess
PRODUCT_COUNT = 50  # Use 20 to reproduce the original comparison group.
from datetime import datetime, timezone
output = work / ('results-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
subprocess.run([sys.executable, str(work / 'm5_sales_experiment.py'), str(parquet),
                '--output-dir', str(output), '--products', str(PRODUCT_COUNT), '--runtime', 'Google Colab'], check=True)
from IPython.display import Markdown, display
display(Markdown((output / 'discussion.md').read_text(encoding='utf-8')))


## Read the results
- RF prediction: average of tree predictions. MAE = mean(abs(actual − predicted)); RMSE = sqrt(mean((actual − predicted)²)).
- Horizon scores compare each product's summed 7/14/28-day quantities. WAPE = sum(abs(error))/sum(actual), undefined with zero total actual. The CSV stores WAPE as a fraction; discussion displays a percentage. Lower quantity errors are better.
- Sale-day positive means actual units > 0; predicted positive means predicted units ≥ validation-selected threshold. Precision = TP/(TP+FP); recall = TP/(TP+FN); F1 = 2TP/(2TP+FP+FN). Undefined metrics are null. F1 and F1 score mean the same thing.
- Compare classification against **always_sale**, and quantity errors against **seasonal_naive** (last week repeated). High F1 is not quantity accuracy or stockout detection.
- Inspect all products, not just favorable examples. A high RMSE relative to MAE suggests large misses. Confirm weaknesses in per-product/day errors before choosing improvements. No operational pass criterion has been agreed.

[Metric definitions](https://scikit-learn.org/stable/modules/model_evaluation.html)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
predictions = pd.read_csv(output / 'daily_predictions.csv', parse_dates=['target_date'])
scores = pd.read_csv(output / 'scores.csv')
display(scores[(scores.split == 'test') & (scores.product_id == 'ALL')])
test = predictions[predictions.split == 'test']
fig, axes = plt.subplots((test.product_id.nunique() + 3) // 4, 4, figsize=(18, 3.2 * ((test.product_id.nunique() + 3) // 4)), sharex=True)
for ax, (pid, group) in zip(axes.flat, test.groupby('product_id')):
    for column in ['actual', 'predicted', 'baseline']:
        ax.plot(group.target_date, group[column], label=column)
    ax.set_title(pid)
    ax.set_ylabel('Units/day')
    ax.tick_params(axis='x', rotation=45)
for ax in axes.flat[test.product_id.nunique():]:
    ax.set_visible(False)
axes.flat[0].legend()
fig.suptitle('Final held-out test: every selected product')
fig.tight_layout()
fig.savefig(output / 'test_predictions.png', dpi=140)
plt.show()


## Save results before the runtime ends
The ZIP contains provenance, selected products/dates, package versions, metrics, per-product scores, held-out daily predictions, history, charts and generated discussion. It contains no production model or Store records. Bring `metrics.json`, `scores.csv` and `discussion.md` back for review before integration. [Colab runtime limitations](https://research.google.com/colaboratory/faq.html).

Later integration: collect Store sales → validate usable history → queued candidate training → chronological evaluation and approval → forecasts after close → advisory replenishment. Existing readiness starts at 56 days (Low), improves at 180 days (Medium), and reaches High at 365+ only with acceptable validation. Having enough rows to fit does not guarantee sufficient history for the full validation protocol.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(output), 'zip', output)
files.download(archive)
